# VISION AI — Entrenamiento de Red Neuronal Convolucional (CNN) con TensorFlow/Keras y Preprocesamiento OpenCV
**Curso:** Machine Learning y Deep Learning (IV Semestre - SENATI)  
**Aplicación directa de las Sesiones 01, 02, 03 y 04:**
1. **Sesión 01:** Escalado de características al rango `[0, 1]` (`MinMaxScaler`) y vector de probabilidades de salida.
2. **Sesión 02:** Hiperparámetros (`epochs`, `batch_size`, `learning_rate`, `Dropout`) y funciones de activación (`ReLU` en capas ocultas para tiempo real a 30 FPS y `Softmax` en la capa de salida).
3. **Sesión 03:** Estructura de Tensores `(Batch, 64, 64, 3)` y arquitectura **CNN (`tf.keras.Sequential`)** con `Conv2D`, `MaxPooling2D`, `Flatten` y `Dense` para inspección de envases (Botella Sana vs. Botella Malograda) y verificación de EPP (Casco y Chaleco).
4. **Sesión 04:** Pipeline de preprocesamiento con **OpenCV (`cv2`)**: conversión a escala de grises ($Y = 0.299R + 0.587G + 0.114B$), `GaussianBlur`, gradiente `Sobel`, `Canny`, `threshold` y recorte de Región de Interés (`ROI`).

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential, layers

print('TensorFlow:', tf.__version__)
print('OpenCV:', cv2.__version__)

## 1. Preprocesamiento Clásico con OpenCV (Sesión 04)
Antes de ingresar el recorte (ROI) a la Red Neuronal Convolucional, estandarizamos la resolución espacial a `(64, 64, 3)` mediante `cv2.resize(..., interpolation=cv2.INTER_AREA)` y normalizamos los tensores en `[0, 1]`.

In [ ]:
def preprocess_roi_opencv(bgr_img: np.ndarray, target_size=(64, 64)) -> np.ndarray:
    # 1. Suavizado espacial Gaussiano para eliminar ruido del sensor (Sesión 04)
    blurred = cv2.GaussianBlur(bgr_img, (3, 3), 0)
    # 2. Normalización espacial a 64x64 píxeles
    resized = cv2.resize(blurred, target_size, interpolation=cv2.INTER_AREA)
    # 3. Conversión de BGR (OpenCV) a RGB y escalado [0, 1]
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    return rgb.astype(np.float32) / 255.0

## 2. Arquitectura de la Red Neuronal Convolucional (CNN - Sesión 03)
Definimos el modelo con `tf.keras.Sequential`:
- **Capas Convolucionales (`Conv2D` + `ReLU`):** Extraen bordes, contornos de tapas, etiquetas y abolladuras.
- **Agrupamiento Máximo (`MaxPooling2D`):** Reduce las dimensiones espaciales reteniendo las activaciones más fuertes.
- **Regularización (`Dropout` - Sesión 02):** Evita el sobreajuste (*overfitting*).
- **Capa de Salida (`Dense` + `Softmax`):** Entrega la distribución de probabilidad entre las 3 clases (`[Botella Conforme, Botella Malograda/Defecto, Etiqueta Desalineada]`).

In [ ]:
# Hiperparámetros (Sesión 02)
IMG_SHAPE = (64, 64, 3)
NUM_CLASSES = 3  # 0: Conforme (Sana), 1: Malograda (Defecto/Sin tapa), 2: Etiqueta ausente/torcida
LEARNING_RATE = 0.001
BATCH_SIZE = 32
EPOCHS = 15

model = Sequential([
    layers.Input(shape=IMG_SHAPE),
    
    # Bloque Convolucional 1
    layers.Conv2D(32, kernel_size=(3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    # Bloque Convolucional 2
    layers.Conv2D(64, kernel_size=(3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    # Bloque Convolucional 3
    layers.Conv2D(128, kernel_size=(3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    # Vectorización y Clasificación Densa
    layers.Flatten(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(NUM_CLASSES, activation='softmax')  # Vector de probabilidades que suma 100%
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()